<a href="https://colab.research.google.com/github/codingbear107/medical_ai/blob/main/colab_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 중앙대 의료AI 해커톤 — Colab Master Notebook

**Stage 1**: PathMNIST로 워밍업 (파이프라인 검증)
**Stage 2**: 본 게임 데이터 도착 후 dataset.py만 교체

## 실행 순서
1. 환경 셋업 + GitHub 클론
2. PyTorch 2.2.0 다운그레이드 (필요 시)
3. PathMNIST 다운로드
4. View A 학습 → View C 학습
5. Merge grid search
6. Few-shot fine-tuning
7. Inference + submission.csv 생성

## 0. 환경 검증

In [9]:
import sys, torch
print(f'Python: {sys.version_info[:3]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}, available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Python: (3, 12, 13)
PyTorch: 2.10.0+cpu
CUDA: None, available: False


## 1. PyTorch 2.2.0 다운그레이드 (이미 했으면 SKIP)

위 셀에서 PyTorch가 2.2.x가 아니면 실행. 끝나면 **런타임 → 세션 다시 시작**.

In [10]:
# 필요할 때만 실행
# !pip uninstall -y torch torchvision torchaudio
# !pip install torch==2.2.0 torchvision==0.17.0 --index-url https://download.pytorch.org/whl/cu121

## 2. GitHub 리포 클론 + 워밍업 패키지 설치

아래 `<USER>/<REPO>`를 본인 GitHub 경로로 수정.

In [11]:
# %cd /content
# !rm -rf medical_ai_hackathon
# !git clone https://github.com/<USER>/medical_ai_hackathon.git
# %cd medical_ai_hackathon
# !pip install -q -r requirements_warmup.txt
import os, sys
PROJECT_DIR = '/content/medical_ai_hackathon'
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
    print('PWD:', os.getcwd())
    print('Files:', os.listdir())

## 3. Google Drive 마운트 (체크포인트 영구 저장용)

In [12]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/medical_ai_ckpts'
os.makedirs(DRIVE_DIR, exist_ok=True)
# checkpoints/ 폴더를 Drive와 symlink (학습 결과 자동 저장)
import os
if not os.path.exists('checkpoints'):
    os.symlink(DRIVE_DIR, 'checkpoints')
print('Checkpoint dir:', os.path.realpath('checkpoints'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint dir: /content/drive/MyDrive/medical_ai_ckpts


## 4. PathMNIST 다운로드 sanity check

In [13]:
%cd /content/medical_ai_hackathon/src
import sys; sys.path.insert(0, '.')
from dataset_pathmnist import _load_pathmnist, create_view_datasets, create_fewshot_datasets
cache = _load_pathmnist()
for split in ['train', 'val', 'test']:
    imgs, labels = cache[split]
    print(f'  {split}: imgs={imgs.shape}, labels unique={len(set(labels.tolist()))}')

[Errno 2] No such file or directory: '/content/medical_ai_hackathon/src'
/content/medical_ai/src
  train: imgs=(89996, 28, 28, 3), labels unique=9
  val: imgs=(10004, 28, 28, 3), labels unique=9
  test: imgs=(7180, 28, 28, 3), labels unique=9


In [14]:
import os, sys

REPO_URL = 'https://github.com/codingbear107/medical_ai.git'
PROJECT_DIR = '/content/medical_ai'

# 옛날 폴더가 있으면 정리
!rm -rf /content/medical_ai_hackathon

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

!pip install -q -r requirements_warmup.txt

print('PWD:', os.getcwd())
print('Files:', sorted(os.listdir()))


Already up to date.
PWD: /content/medical_ai
Files: ['.git', '.gitignore', 'README.md', 'colab_main.ipynb', 'requirements_submit.txt', 'requirements_warmup.txt', 'src']


In [15]:
import os, sys

PROJECT_DIR = '/content/medical_ai'

# 옛 폴더 정리
!rm -rf /content/medical_ai_hackathon /content/medical_ai

# Public 클론
!git clone https://github.com/codingbear107/medical_ai.git {PROJECT_DIR}

os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

!pip install -q -r requirements_warmup.txt

print('\nPWD:', os.getcwd())
print('Files:', sorted(os.listdir()))
print('src/:', sorted(os.listdir('src')))


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/medical_ai'...
fatal: Unable to read current working directory: No such file or directory


FileNotFoundError: [Errno 2] No such file or directory: '/content/medical_ai'

In [ ]:
%cd /content/medical_ai/src
import sys; sys.path.insert(0, '.')
from dataset_pathmnist import _load_pathmnist, create_view_datasets, create_fewshot_datasets
cache = _load_pathmnist()
for split in ['train', 'val', 'test']:
    imgs, labels = cache[split]
    print(f'  {split}: imgs={imgs.shape}, labels unique={len(set(labels.tolist()))}')


## 5. Phase 1 — View A 학습

epochs를 줄여서 빠르게 한 번 돌려보고, 잘 되면 늘리세요. 첫 실행은 epochs=20 정도 추천.

In [16]:
!python train_view.py --view A --dataset pathmnist --epochs 20 --seed 42 --save_init

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
python3: can't open file 'train_view.py': [Errno 2] No such file or directory


In [17]:
import os, sys, subprocess

# 1. cwd 강제 복구
os.chdir('/')
os.chdir('/content')
print('cwd 복구 후:', os.getcwd())

# 2. 망가진 폴더 정리
subprocess.run(['rm', '-rf', '/content/medical_ai_hackathon', '/content/medical_ai'])

# 3. 클론
ret = subprocess.run(
    ['git', 'clone', 'https://github.com/codingbear107/medical_ai.git', '/content/medical_ai'],
    capture_output=True, text=True
)
print('clone stdout:', ret.stdout)
print('clone stderr:', ret.stderr)

# 4. 진입 + sys.path
os.chdir('/content/medical_ai/src')
if '/content/medical_ai/src' not in sys.path:
    sys.path.insert(0, '/content/medical_ai/src')

print('\n현재 cwd:', os.getcwd())
print('src/ files:', sorted(os.listdir('.')))


cwd 복구 후: /content
clone stdout: 
clone stderr: Cloning into '/content/medical_ai'...


현재 cwd: /content/medical_ai/src
src/ files: ['__init__.py', 'augmentations.py', 'config.py', 'dataset.py', 'dataset_pathmnist.py', 'ewc.py', 'inference.py', 'merge.py', 'merge_and_eval.py', 'model.py', 'train_fewshot.py', 'train_view.py', 'utils.py']


In [18]:
!pip install -q medmnist

from dataset_pathmnist import _load_pathmnist
cache = _load_pathmnist()
for split in ['train', 'val', 'test']:
    imgs, labels = cache[split]
    print(f'  {split}: imgs={imgs.shape}')


  train: imgs=(89996, 28, 28, 3)
  val: imgs=(10004, 28, 28, 3)
  test: imgs=(7180, 28, 28, 3)


In [1]:
import os
os.chdir('/content/medical_ai/src')
!pwd && ls train_view.py
!python train_view.py --view A --dataset pathmnist --epochs 20 --seed 42 --save_init


FileNotFoundError: [Errno 2] No such file or directory: '/content/medical_ai/src'

In [ ]:
import os, sys, subprocess, torch

# 1. GPU 확인 — GPU 아니면 즉시 중단
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임 아님! 런타임 → 유형 변경 → T4 GPU 후 재시작")
print('GPU:', torch.cuda.get_device_name(0))

# 2. 클론
os.chdir('/content')
subprocess.run(['rm', '-rf', '/content/medical_ai'])
subprocess.run(['git', 'clone',
                'https://github.com/codingbear107/medical_ai.git',
                '/content/medical_ai'], check=True)

# 3. 진입
os.chdir('/content/medical_ai/src')
sys.path.insert(0, '/content/medical_ai/src')
print('PWD:', os.getcwd())
print('Files:', sorted(os.listdir()))

# 4. 패키지
!pip install -q medmnist

# 5. View A 학습
!python train_view.py --view A --dataset pathmnist --epochs 20 --seed 42 --save_init


CUDA: True
GPU: Tesla T4
PWD: /content/medical_ai/src
Files: ['__init__.py', 'augmentations.py', 'config.py', 'dataset.py', 'dataset_pathmnist.py', 'ewc.py', 'inference.py', 'merge.py', 'merge_and_eval.py', 'model.py', 'train_fewshot.py', 'train_view.py', 'utils.py']
[train_view] view=A, dataset=pathmnist, seed=42, device=cuda
[train_view] num_classes=9
[train_view] train=89996, val=10004
[train_view] params: total=2,776,265, trainable=2,776,265
[train_view] init state 저장: /content/medical_ai/checkpoints/init.pth
[view=A] Epoch   1/20 | Loss: 1.7236 | Val F1: 0.5441 | Acc: 0.5402 | LR: 9.97e-04
  -> Best 저장: /content/medical_ai/checkpoints/axial_model.pth (F1=0.5441)
[view=A] Epoch   2/20 | Loss: 1.4745 | Val F1: 0.7469 | Acc: 0.7542 | LR: 9.89e-04
  -> Best 저장: /content/medical_ai/checkpoints/axial_model.pth (F1=0.7469)
[view=A] Epoch   3/20 | Loss: 1.3793 | Val F1: 0.7229 | Acc: 0.7293 | LR: 9.76e-04


## 6. Phase 1 — View C 학습

In [ ]:
!python train_view.py --view C --dataset pathmnist --epochs 20 --seed 42

## 7. Phase 2 — Merge Grid Search

4가지 merge 기법 + 여러 λ를 비교. Sagittal validation F1으로 best 선정 → base_model.pth 저장.

In [ ]:
!python merge_and_eval.py --dataset pathmnist --seed 42

## 8. Phase 3 — Few-shot Fine-tuning on Sagittal

In [ ]:
!python train_fewshot.py --dataset pathmnist --seed 42 --epochs 30

## 9. Inference (sanity check)

PathMNIST에는 별도 test 폴더가 없으니, 워밍업에선 이 셀을 실행하지 않아도 됩니다.
본 게임 단계에서 활용.

In [ ]:
# 본 게임 데이터 도착 후:
# !python inference.py --model_path checkpoints/model.pth --test_dir /content/data/test --output submission.csv
# !head submission.csv

## 10. 결과 확인 + 다음 실험

- `checkpoints/merge_grid_results.csv` 에 grid search 결과 정리됨
- 어떤 merge 기법이 가장 좋았는지 확인
- λ_EWC 등 하이퍼파라미터 조정해서 재실행

In [ ]:
import pandas as pd
df = pd.read_csv('checkpoints/merge_grid_results.csv')
print(df.sort_values('sag_f1', ascending=False).to_string(index=False))